# Model Notebook
This notebook is intended for building and training the malaria model.


In [3]:
# Import necessary libraries
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler    
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC

In [4]:
# Load the dataset
df = pd.read_csv('cleaned_malaria.csv')
# Display the first few rows of the dataset 
print(df.head())

# Encode the gender and underlying issues
from sklearn.preprocessing import LabelEncoder
label_encoder = LabelEncoder()
categorical_columns = ['GENDER', 'Underlying Issues']
for column in categorical_columns:
    df[column] = label_encoder.fit_transform(df[column])

# Display the first few rows of the dataset after encoding
print(df.head())

   AGE GENDER                                           SYMPTOMS  WEIGHT(kg)  \
0   23      M                   cough , headache, Body weakness    72.700000   
1   18      F  Abdominal pain ,headache,Dizzinesss,vomitting,...   52.800000   
2   51      F  Chest pain, headache, Body pain, Bitter taste ...   64.116154   
3   18      F             mouthsore, Catarrh, itching, Body pain   50.000000   
4   19      F             Headache, vomitting, loss of appetite    88.500000   

   TEMPERATURE(degreec) Underlying Issues PRESCRIBED MEDICATION  
0                  36.4              none            ACT tablet  
1                  35.6              none            ACT tablet  
2                  36.4              none            ACT tablet  
3                  36.6              none            ACT tablet  
4                  36.0              none            ACT tablet  
   AGE  GENDER                                           SYMPTOMS  WEIGHT(kg)  \
0   23       4                   cough , h

In [5]:
# Standardize the numerical columns
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
numerical_columns = ['AGE', 'WEIGHT(kg)', 'TEMPERATURE(degreec)']
df[numerical_columns] = scaler.fit_transform(df[numerical_columns])

In [7]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler, MultiLabelBinarizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.multioutput import MultiOutputClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, hamming_loss
from tabulate import tabulate

# Load dataset
df = pd.read_csv('cleaned_malaria.csv')

# Display the first few rows
print("\n📋 Initial Dataset:")
print(tabulate(df.head(), headers='keys', tablefmt='grid'))

# Encode categorical columns (GENDER, Underlying Issues)
label_encoder = LabelEncoder()
categorical_columns = ['GENDER', 'Underlying Issues']
for column in categorical_columns:
    df[column] = label_encoder.fit_transform(df[column].astype(str))

# Show after encoding
print("\n🔢 After Encoding Categorical Columns:")
print(tabulate(df.head(), headers='keys', tablefmt='fancy_grid'))

# Standardize numerical columns
scaler = StandardScaler()
numerical_columns = ['AGE', 'WEIGHT(kg)', 'TEMPERATURE(degreec)']
df[numerical_columns] = scaler.fit_transform(df[numerical_columns])

# TF-IDF vectorization for SYMPTOMS column (features)
tfidf_vectorizer = TfidfVectorizer(stop_words='english', max_features=100)
X = tfidf_vectorizer.fit_transform(df['SYMPTOMS'].fillna(''))
print("\n🔠 SYMPTOMS TF-IDF shape:", X.shape)

# Multi-label binarization for PRESCRIBED MEDICATION column (targets)
mlb = MultiLabelBinarizer()
y = mlb.fit_transform(df['PRESCRIBED MEDICATION'].fillna('').str.lower().str.split(','))
print("💊 PRESCRIBED MEDICATION Binary Shape:", y.shape)
print("📚 Medications List:", mlb.classes_)

# Display the first few rows
print("\n📋 Initial Dataset:")
print(tabulate(df.head(), headers='keys', tablefmt='grid'))

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Multi-output classifier
classifier = MultiOutputClassifier(RandomForestClassifier(random_state=42))
classifier.fit(X_train, y_train)

# Predict
y_pred = classifier.predict(X_test)

# Evaluation
print("\n✅ Prediction shape:", y_pred.shape)
print("📉 Hamming Loss:", hamming_loss(y_test, y_pred))
print("\n📊 Classification Report:")
print(classification_report(y_test, y_pred, target_names=mlb.classes_, zero_division=0))



📋 Initial Dataset:
+----+-------+----------+-------------------------------------------------------------------------------+--------------+------------------------+---------------------+-------------------------+
|    |   AGE | GENDER   | SYMPTOMS                                                                      |   WEIGHT(kg) |   TEMPERATURE(degreec) | Underlying Issues   | PRESCRIBED MEDICATION   |
+====+=======+==========+===============================================================================+==============+========================+=====================+=========================+
|  0 |    23 | M        | cough , headache, Body weakness                                               |      72.7    |                   36.4 | none                | ACT tablet              |
+----+-------+----------+-------------------------------------------------------------------------------+--------------+------------------------+---------------------+-------------------------+
|  1 |    

#another way

In [34]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from scipy.sparse import hstack

# Load data
df = pd.read_csv('cleaned_malaria.csv')

# Handle missing data (if any)
df.fillna('', inplace=True)

# Encode categorical columns
gender_enc = LabelEncoder()
issue_enc = LabelEncoder()
med_enc = LabelEncoder()

df['GENDER'] = gender_enc.fit_transform(df['GENDER'])
df['Underlying Issues'] = issue_enc.fit_transform(df['Underlying Issues'])
df['PRESCRIBED MEDICATION'] = med_enc.fit_transform(df['PRESCRIBED MEDICATION'].str.strip())

# Scale numerical features
scaler = StandardScaler()
numeric_features = scaler.fit_transform(df[['AGE', 'WEIGHT(kg)', 'TEMPERATURE(degreec)']])

# TF-IDF for symptoms
tfidf = TfidfVectorizer(stop_words='english', max_features=100)
symptoms_tfidf = tfidf.fit_transform(df['SYMPTOMS'].str.lower())

# Combine all features
from scipy.sparse import csr_matrix
other_features = csr_matrix(df[['GENDER', 'Underlying Issues']].values)
X = hstack([symptoms_tfidf, csr_matrix(numeric_features), other_features])
y = df['PRESCRIBED MEDICATION']

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Model
clf = RandomForestClassifier(random_state=42)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

# Report
from sklearn.utils.multiclass import unique_labels

# Get only the labels that exist in y_test or y_pred
present_labels = unique_labels(y_test, y_pred)

# Map back to class names using the LabelEncoder
present_class_names = med_enc.inverse_transform(present_labels)

# Generate classification report
print(classification_report(y_test, y_pred, labels=present_labels, target_names=present_class_names))



                                                             precision    recall  f1-score   support

                       ACT tab, I.M Arthemeter Lumefantrine       0.00      0.00      0.00         1
                                                 ACT tablet       0.50      0.08      0.14        12
                             Arthemeter Lumefantrine tablet       0.47      0.84      0.61        45
                                                 Camosunate       0.00      0.00      0.00         2
                                           Emal, ACT tablet       0.00      0.00      0.00         1
                                                   Fansidar       0.00      0.00      0.00         2
                                I.M Arthemeter Lumefantrine       0.33      0.50      0.40         4
                       I.M Arthemeter Lumefantrine ,ACT tab       0.00      0.00      0.00         0
I.M Arthemeter Lumefantrine ,Arthemeter Lumefantrine tablet       0.00      0.00      0.00

c:\Users\FALOWO PC\Downloads\malaria-drug-recommender-system\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1706: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\FALOWO PC\Downloads\malaria-drug-recommender-system\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1706: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\FALOWO PC\Downloads\malaria-drug-recommender-system\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1706: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control th

In [39]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.utils.multiclass import unique_labels
from scipy.sparse import hstack, csr_matrix
from imblearn.over_sampling import SMOTE

# Load data
df = pd.read_csv('cleaned_malaria.csv')

# Handle missing data
df.fillna('', inplace=True)

# Encode categorical columns
gender_enc = LabelEncoder()
issue_enc = LabelEncoder()
med_enc = LabelEncoder()

df['GENDER'] = gender_enc.fit_transform(df['GENDER'])
df['Underlying Issues'] = issue_enc.fit_transform(df['Underlying Issues'])
df['PRESCRIBED MEDICATION'] = med_enc.fit_transform(df['PRESCRIBED MEDICATION'].str.strip())

# Scale numerical features
scaler = StandardScaler()
numeric_features = scaler.fit_transform(df[['AGE', 'WEIGHT(kg)', 'TEMPERATURE(degreec)']])

# TF-IDF for SYMPTOMS
tfidf = TfidfVectorizer(stop_words='english', max_features=100)
symptoms_tfidf = tfidf.fit_transform(df['SYMPTOMS'].str.lower())

# Combine features
other_features = csr_matrix(df[['GENDER', 'Underlying Issues']].values)
X = hstack([symptoms_tfidf, csr_matrix(numeric_features), other_features])
y = df['PRESCRIBED MEDICATION']

# Convert to dense before SMOTE
X_dense = X.toarray()

# Split first
X_train, X_test, y_train, y_test = train_test_split(X_dense, y, test_size=0.2, random_state=42, stratify=y)

# Apply SMOTE
sm = SMOTE(random_state=42)
X_resampled, y_resampled = sm.fit_resample(X_train, y_train)

# ⏬ Optional: Preview SMOTE-resampled training data
resampled_df = pd.DataFrame(X_resampled)
resampled_df['PRESCRIBED MEDICATION'] = med_enc.inverse_transform(y_resampled)
print("\nResampled Training Data Preview (first 5 rows):")
print(resampled_df.head())

# Train model on resampled data
clf = RandomForestClassifier(random_state=42)
clf.fit(X_resampled, y_resampled)
y_pred = clf.predict(X_test)

# Report
present_labels = unique_labels(y_test, y_pred)
present_class_names = med_enc.inverse_transform(present_labels)

print("\nClassification Report:")
print(classification_report(y_test, y_pred, labels=present_labels, target_names=present_class_names))


ImportError: cannot import name '_get_column_indices' from 'sklearn.utils' (c:\Users\FALOWO PC\Downloads\malaria-drug-recommender-system\.venv\Lib\site-packages\sklearn\utils\__init__.py)